# §1 
## Assumption
 - u : up factor
 - d : down factor
 - r : intrest rate
 - non-arbitrage:  0 < d < 1 + r < u

In [1]:
def binomial_option(S0, u, d, r, K):
    S_up = S0 * u
    S_down = S0 * d

    payoff_up = max(0, S_up - K)
    payoff_down = max(0, S_down - K)

    delta = (payoff_up - payoff_down) / (S_up - S_down)
    B = (payoff_down - delta * S_down) / (1 + r)

    price = delta * S0 + B

    return price

## Assumptions on the function above
 - shares of stock can be subdivided for sale or purchase
 - the interest rate for investing is the same as the interest rate for borrowing
 -the purchase price of stock is the same as the selling price(i.e., there is zero bid_ask spread)
 - at any time, the stock can take only two possible values in the next period
All these assumption except the last also underlie the Black-Scholes-Merton option-pricing formula. 

In [2]:
def binomial_option_price(S0,u, d, r, K):
    """
    One-step binomial option pricing (call option)
    
    S0: initial stock price
    u: up factor
    d: down factor
    r: risk-free interest rate
    K: strike price
    """
    # future stock prices
    S_up = S0 * u
    S_down = S0 * d

    # payoff
    V_up = max(S_up - K, 0)
    V_down = max(S_down - K, 0)

    # replication 
    delta = (V_up - V_down) / (S_up - S_down)
    B = (V_down - delta * S_down) / (1 + r)

    # option price
    X0 = delta * S0 + B
    
    return X0

## One-Period Binomial Model – Replication

We construct a portfolio consisting of stock and a money market account:

$$
X_1 = \Delta_0 S_1 + (1+r)(X_0 - \Delta_0 S_0)
= (1+r)X_0 + \Delta_0 \big(S_1 - (1+r)S_0\big)
$$

We choose $X_0$ and $\Delta_0$ such that

$$
X_1(H) = V_1(H), \quad X_1(T) = V_1(T)
$$

This leads to:

$$
X_0 + \Delta_0 \left(\frac{1}{1+r} S_1(H) - S_0\right)
= \frac{1}{1+r} V_1(H)
$$

$$
X_0 + \Delta_0 \left(\frac{1}{1+r} S_1(T) - S_0\right)
= \frac{1}{1+r} V_1(T)
$$

---

### Risk-neutral probability

Multiply the first equation by $\tilde{p}$ and the second by $\tilde{q} = 1 - \tilde{p}$:

$$
X_0 + \Delta_0 \left(
\frac{1}{1+r} \big[\tilde{p} S_1(H) + \tilde{q} S_1(T)\big] - S_0
\right)
=
\frac{1}{1+r} \big[\tilde{p} V_1(H) + \tilde{q} V_1(T)\big]
$$

Choose $\tilde{p}$ such that:

$$
S_0 = \frac{1}{1+r} \big[\tilde{p} S_1(H) + \tilde{q} S_1(T)\big]
$$

Then the $\Delta_0$ term disappears, and we obtain:

$$
X_0 = \frac{1}{1+r} \big[\tilde{p} V_1(H) + \tilde{q} V_1(T)\big]
$$

---

### Risk-neutral probabilities

$$
\tilde{p} = \frac{1+r-d}{u-d}, \quad
\tilde{q} = \frac{u-1-r}{u-d}
$$

---

### Delta hedging

$$
\Delta_0 = \frac{V_1(H) - V_1(T)}{S_1(H) - S_1(T)}
$$

In [4]:
def binomial_full(S0, u, d, r, K):
    S_up = S0 * u
    S_down = S0 * d
    V_up = max(S_up - K, 0)
    V_down = max(S_down - K, 0)

    delta = (V_up - V_down) / (S_up - S_down)
    B = (V_down - delta * S_down) / (1 + r)
    X0 = delta * S0 + B
    #resik-neutral probability
    p = (1 + r - d) / (u - d)
    X0_rn = (1 / (1 + r) )* (p * V_up + (1 - p) * V_down)

    return {
        "price": X0,
        "delta": delta,
        "bond": B,
        "p": p,
        "price_rn": X0_rn
    }

In [6]:
price = binomial_option_price(4, 2, 0.5, 0.25, 5)
print(price)
binomial_full(4, 2, 0.5, 0.25, 5)

1.2


{'price': 1.2,
 'delta': 0.5,
 'bond': -0.8,
 'p': 0.5,
 'price_rn': 1.2000000000000002}

## Theorem 1.2.2 (Replication in the multiperiod binomial model)

Consider an $N$-period binomial asset-pricing model, with

$$
0 < d < 1 + r < u
$$

and

$$
\tilde{p} = \frac{1 + r - d}{u - d}, \quad 
\tilde{q} = \frac{u - 1 - r}{u - d}.
$$

Define recursively:

$$
V_n(\omega_1 \cdots \omega_n)
=
\frac{1}{1+r}
\left[
\tilde{p} V_{n+1}(\omega_1 \cdots \omega_n H)
+
\tilde{q} V_{n+1}(\omega_1 \cdots \omega_n T)
\right]
$$

$$
\Delta_n(\omega_1 \cdots \omega_n)
=
\frac{
V_{n+1}(\omega_1 \cdots \omega_n H)
-
V_{n+1}(\omega_1 \cdots \omega_n T)
}{
S_{n+1}(\omega_1 \cdots \omega_n H)
-
S_{n+1}(\omega_1 \cdots \omega_n T)
}
$$

In [1]:
def binomial_option_price(S0, u, d, r, K, N):
    """
    Multiperiod binomial model (European call option)

    S0: initial stock price
    u: up factor
    d: down factor
    r: interest rate
    K; strike price
    N: number of periods
    """

    # Risk-neutral probability
    p = (1 + r - d) / (u - d)

    # Step 1: Build stock price tree
    stock_tree = []
    for n in range(N + 1):
        level = []
        for i in range(n + 1):
            price = S0 * (u ** i) * (d ** (n - i))
            level.append(price)
        stock_tree.append(level)
    
    # Step 2: Payoff at maturity
    option_values = [max(S - K, 0) for S in stock_tree[N]]

    # Step 3: Backward induciton
    for n in range(N - 1, -1, -1):
        new_values = []
        for i in range(n + 1):
            value = (1 / (1 + r)) * (
                p * option_values[i + 1] + (1 - p) * option_values[i]
            )
            new_values.append(value)
        option_values = new_values
    
    return option_values[0]

In [10]:
binomial_option_price(4, 2, 1/2, 1/4, 34, 2)

0.0

In [8]:
def binomial_tree_with_paths(S0, u, d, r, K, N):
    q = (1 + r - d) / (u - d)

    S = {"": S0}
    
    for n in range(1, N + 1):
        new_S = {}
        for path, price in S.items():
            new_S[path + "H"] = price * u
            new_S[path + "T"] = price * d
        S = new_S

    V = {}
    for path, price in S.items():
        V[path] = max(price - K, 0)

    all_V = {N: V.copy()}

    for n in range(N - 1, -1, -1):
        new_V = {}
        for path in set(p[:-1] for p in V.keys()):
            VH = V[path + "H"]
            VT = V[path + "T"]

            new_V[path] = (1 / (1 + r)) * (q * VH + (1 - q) * VT)

        V = new_V
        all_V[n] = V.copy()

    return all_V

In [13]:
tree = binomial_tree_with_paths(4, 2, 1/2, 1/4, 5, 2)
print(tree)

{2: {'HH': 11, 'HT': 0, 'TH': 0, 'TT': 0}, 1: {'H': 4.4, 'T': 0.0}, 0: {'': 1.7600000000000002}}
